# Customer Churn Prediction
## Group 4 – Machine Learning
### Member 1 – Logistic Regression & ML Foundation

**Objective:**  
Develop the common machine learning foundation and build a Logistic Regression churn classifier using the finalized feature-engineered dataset from Group 3.

This notebook covers:
- Dataset handoff validation
- Feature and target definition
- Train/test methodology
- Preprocessing pipeline
- Dummy baseline
- Logistic Regression
- Cross-validation
- Hyperparameter tuning
- Model evaluation
- Member 1 model handoff

In [1]:
# ==========================================
# 1. Imports and Configuration
# ==========================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import sklearn

RANDOM_STATE = 42

print("Python executable   :", sys.executable)
print("Python version      :", sys.version.split()[0])
print("NumPy version       :", np.__version__)
print("Pandas version      :", pd.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("Random state        :", RANDOM_STATE)

Python executable   : d:\Internship Tasks\Group Task - Customer Churn Prediction & Business Intelligence System\Customer-Churn-Prediction-Business-Intelligence-System\.venv\Scripts\python.exe
Python version      : 3.13.3
NumPy version       : 2.5.2
Pandas version      : 3.0.5
Scikit-learn version: 1.9.0
Random state        : 42


## 2. Load Group 3 Feature-Engineered Dataset

The modelling dataset is taken from the final feature-engineering output produced by Group 3.

Model preprocessing will be fitted later using training data only to avoid preprocessing leakage.

In [2]:
# ==========================================
# 2. Locate Project Root and Dataset
# ==========================================

DATA_RELATIVE_PATH = (
    Path("Feature Engineering & Statistical Analysis")
    / "outputs"
    / "customer_churn_feature_engineered.csv"
)

def find_project_root(start_path=None):
    """
    Search the current directory and its parent directories
    until the Group 3 modelling dataset is found.
    """
    current = Path(start_path or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        dataset_path = candidate / DATA_RELATIVE_PATH

        if dataset_path.exists():
            return candidate

    raise FileNotFoundError(
        "Project root could not be located. "
        "Expected to find the Group 3 feature-engineered dataset."
    )


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / DATA_RELATIVE_PATH

print("Project root:")
print(PROJECT_ROOT)

print("\nDataset:")
print(DATA_PATH)

Project root:
D:\Internship Tasks\Group Task - Customer Churn Prediction & Business Intelligence System\Customer-Churn-Prediction-Business-Intelligence-System

Dataset:
D:\Internship Tasks\Group Task - Customer Churn Prediction & Business Intelligence System\Customer-Churn-Prediction-Business-Intelligence-System\Feature Engineering & Statistical Analysis\outputs\customer_churn_feature_engineered.csv


In [3]:
# ==========================================
# 3. Load Dataset
# ==========================================

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

Dataset loaded successfully.
Rows    : 7,043
Columns : 29


In [4]:
df.head()

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,...,Churn Value,Is_New_Customer,Tenure_Band,Is_Month_to_Month,High_Monthly_Charge,Fiber_Monthly_Risk,New_High_Spend,New_Monthly_Customer,Support_Service_Count,Security_Tech_Bundle
0,Male,0,0,0,2,1,0,DSL,1,1,...,1,1,0-6,1,0,0,0,1,2,0
1,Female,0,0,1,2,1,0,Fiber optic,0,0,...,1,1,0-6,1,1,1,1,1,0,0
2,Female,0,0,1,8,1,1,Fiber optic,0,0,...,1,1,7-12,1,1,1,1,1,1,0
3,Female,0,1,1,28,1,1,Fiber optic,0,0,...,1,0,25-48,1,1,1,0,0,2,0
4,Male,0,0,1,49,1,1,Fiber optic,0,1,...,1,0,49+,1,1,1,0,0,2,0


In [5]:
# ==========================================
# 4. Dataset Structure
# ==========================================

print("Columns:\n")

for number, column in enumerate(df.columns, start=1):
    print(f"{number:02d}. {column}")

print("\nDataset information:\n")
df.info()

Columns:

01. Gender
02. Senior Citizen
03. Partner
04. Dependents
05. Tenure Months
06. Phone Service
07. Multiple Lines
08. Internet Service
09. Online Security
10. Online Backup
11. Device Protection
12. Tech Support
13. Streaming TV
14. Streaming Movies
15. Contract
16. Paperless Billing
17. Payment Method
18. Monthly Charges
19. Total Charges
20. Churn Value
21. Is_New_Customer
22. Tenure_Band
23. Is_Month_to_Month
24. High_Monthly_Charge
25. Fiber_Monthly_Risk
26. New_High_Spend
27. New_Monthly_Customer
28. Support_Service_Count
29. Security_Tech_Bundle

Dataset information:

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 29 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Gender                 7043 non-null   str    
 1   Senior Citizen         7043 non-null   int64  
 2   Partner                7043 non-null   int64  
 3   Dependents             7043 non-null   int64  
 4 

## 3. Group 3 Data Handoff Validation

Before modelling, the dataset is checked for missing values, target integrity, required features and potential target leakage.

In [6]:
# ==========================================
# 5. Missing Value Check
# ==========================================

missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]

if missing_values.empty:
    print("No missing values detected.")
else:
    print("Missing values detected:")
    print(missing_values)

No missing values detected.


In [7]:
# ==========================================
# 6. Target Validation
# ==========================================

TARGET = "Churn Value"

if TARGET not in df.columns:
    raise ValueError(
        f"Target column '{TARGET}' was not found."
    )

print("Target:", TARGET)

print("\nUnique values:")
print(sorted(df[TARGET].dropna().unique()))

print("\nTarget counts:")
print(df[TARGET].value_counts().sort_index())

Target: Churn Value

Unique values:
[np.int64(0), np.int64(1)]

Target counts:
Churn Value
0    5174
1    1869
Name: count, dtype: int64


In [8]:
expected_target_values = {0, 1}
actual_target_values = set(df[TARGET].dropna().unique())

if actual_target_values != expected_target_values:
    raise ValueError(
        f"Unexpected target values: {actual_target_values}"
    )

print("\nTarget validation passed: binary target {0, 1}.")


Target validation passed: binary target {0, 1}.


In [9]:
# ==========================================
# 7. Leakage Protection
# ==========================================

LEAKAGE_COLUMNS = [
    "Churn Label",
    "Churn Score",
    "Churn Reason"
]

present_leakage_columns = [
    column
    for column in LEAKAGE_COLUMNS
    if column in df.columns
]

if present_leakage_columns:
    print("WARNING: Leakage-related columns exist:")
    print(present_leakage_columns)
else:
    print("No known leakage columns are present.")

No known leakage columns are present.


In [10]:
# ==========================================
# 8. Group 3 Recommended Core Features
# ==========================================

CORE_FEATURES = [
    "Senior Citizen",
    "Partner",
    "Dependents",
    "Tenure Months",
    "Multiple Lines",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Contract",
    "Paperless Billing",
    "Payment Method",
    "Monthly Charges",
    "Total Charges",
    "Fiber_Monthly_Risk",
    "New_High_Spend",
    "New_Monthly_Customer",
    "Security_Tech_Bundle"
]

print("Number of core features:", len(CORE_FEATURES))

Number of core features: 21


In [11]:
missing_core_features = [
    feature
    for feature in CORE_FEATURES
    if feature not in df.columns
]

if missing_core_features:
    raise ValueError(
        f"Missing required features: {missing_core_features}"
    )

print("All 21 Group 3 core features are available.")

All 21 Group 3 core features are available.


In [12]:
leakage_in_features = set(CORE_FEATURES).intersection(
    LEAKAGE_COLUMNS
)

if leakage_in_features:
    raise ValueError(
        f"Leakage columns detected in feature list: "
        f"{leakage_in_features}"
    )

print("Feature leakage check passed.")

Feature leakage check passed.


In [13]:
# ==========================================
# 9. Predictors and Target
# ==========================================

X = df[CORE_FEATURES].copy()
y = df[TARGET].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 21)
y shape: (7043,)


In [14]:
# ==========================================
# 10. Class Distribution
# ==========================================

class_counts = y.value_counts().sort_index()

class_percentages = (
    y.value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

class_distribution = pd.DataFrame({
    "Count": class_counts,
    "Percentage (%)": class_percentages.round(2)
})

class_distribution.index = [
    "Non-Churn (0)",
    "Churn (1)"
]

class_distribution

,Count,Percentage (%)
Non-Churn (0),5174,73.46
Churn (1),1869,26.54


### Class Distribution Interpretation

The churn target is moderately imbalanced, with non-churn customers forming the majority class.

Therefore, accuracy alone is not sufficient for model evaluation. A classifier may achieve relatively high accuracy by favouring the majority class while failing to identify customers who actually churn.

For this reason, the modelling stage will also evaluate:

- Precision
- Recall
- F1-score
- ROC-AUC

Stratification will be used during train/test splitting and cross-validation to preserve the churn distribution.

In [15]:
# ==========================================
# 11. Duplicate Row Inspection
# ==========================================

duplicate_rows = df.duplicated().sum()

print(
    "Exact duplicate rows in modelling dataset:",
    duplicate_rows
)

Exact duplicate rows in modelling dataset: 26


### Duplicate Handling Note

Duplicate-looking modelling rows are not removed automatically.

After customer identifiers and location-related attributes are excluded, different customers can legitimately share the same available predictor values. Therefore, duplicated predictor profiles do not necessarily represent duplicated customer records.

In [16]:
# ==========================================
# 12. Data Handoff Validation Summary
# ==========================================

print("=" * 60)
print("GROUP 3 -> GROUP 4 DATA HANDOFF VALIDATION")
print("=" * 60)

print(f"Dataset rows              : {df.shape[0]:,}")
print(f"Dataset columns           : {df.shape[1]}")
print(f"Selected core features    : {len(CORE_FEATURES)}")
print(f"Target                    : {TARGET}")

print(
    f"Missing values in X       : "
    f"{X.isnull().sum().sum()}"
)

print(
    f"Missing values in y       : "
    f"{y.isnull().sum()}"
)

print(
    f"Leakage columns in features: "
    f"{len(leakage_in_features)}"
)

print(
    f"Non-churn customers       : "
    f"{(y == 0).sum():,}"
)

print(
    f"Churn customers           : "
    f"{(y == 1).sum():,}"
)

print("=" * 60)
print("DATA READY FOR TRAIN/TEST SPLITTING")
print("=" * 60)

GROUP 3 -> GROUP 4 DATA HANDOFF VALIDATION
Dataset rows              : 7,043
Dataset columns           : 29
Selected core features    : 21
Target                    : Churn Value
Missing values in X       : 0
Missing values in y       : 0
Leakage columns in features: 0
Non-churn customers       : 5,174
Churn customers           : 1,869
DATA READY FOR TRAIN/TEST SPLITTING


## 4. Train/Test Split and Cross-Validation Strategy

The dataset is divided into training and held-out test sets before model preprocessing or training.

An 80/20 stratified split is used so that both subsets preserve the original churn class distribution.

The held-out test set will not be used for model tuning or model selection. Model development will be performed using the training set with 5-fold stratified cross-validation.

In [17]:
# ==========================================
# 13. Train/Test and CV Imports
# ==========================================

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

In [18]:
# ==========================================
# 14. Common Experimental Configuration
# ==========================================

TEST_SIZE = 0.20
N_SPLITS = 5

print("Test size        :", TEST_SIZE)
print("Random state     :", RANDOM_STATE)
print("CV folds         :", N_SPLITS)

Test size        : 0.2
Random state     : 42
CV folds         : 5


In [19]:
# ==========================================
# 15. Stratified Train/Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train/test split completed.")
print()
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

Train/test split completed.

X_train: (5634, 21)
X_test : (1409, 21)
y_train: (5634,)
y_test : (1409,)


In [20]:
# ==========================================
# 16. Verify Class Distribution
# ==========================================

def class_distribution(series, name):
    counts = series.value_counts().sort_index()

    percentages = (
        series.value_counts(normalize=True)
        .sort_index()
        .mul(100)
    )

    return pd.DataFrame({
        "Dataset": name,
        "Class": counts.index,
        "Count": counts.values,
        "Percentage (%)": percentages.values.round(2)
    })


distribution_comparison = pd.concat(
    [
        class_distribution(y, "Full Dataset"),
        class_distribution(y_train, "Training Set"),
        class_distribution(y_test, "Test Set"),
    ],
    ignore_index=True
)

distribution_comparison

,Dataset,Class,Count,Percentage (%)
0,Full Dataset,0,5174,73.46
1,Full Dataset,1,1869,26.54
2,Training Set,0,4139,73.46
3,Training Set,1,1495,26.54
4,Test Set,0,1035,73.46
5,Test Set,1,374,26.54


### Stratification Check

The training and test sets preserve approximately the same churn distribution as the complete dataset.

This is important because churn is the minority class. Stratification prevents one subset from receiving an unusually high or low proportion of churn customers and therefore improves the reliability of model comparison.

In [21]:
# ==========================================
# 17. Train/Test Overlap Check
# ==========================================

train_indices = set(X_train.index)
test_indices = set(X_test.index)

overlap = train_indices.intersection(test_indices)

print("Training rows :", len(train_indices))
print("Test rows     :", len(test_indices))
print("Overlap count :", len(overlap))

if overlap:
    raise ValueError(
        "Train/test overlap detected."
    )

print("Train/test separation check passed.")

Training rows : 5634
Test rows     : 1409
Overlap count : 0
Train/test separation check passed.


In [22]:
# ==========================================
# 18. Split Integrity Check
# ==========================================

total_split_rows = len(X_train) + len(X_test)

print("Original rows :", len(X))
print("Split rows    :", total_split_rows)

if total_split_rows != len(X):
    raise ValueError(
        "Train/test split row count does not match original data."
    )

print("Split integrity check passed.")

Original rows : 7043
Split rows    : 7043
Split integrity check passed.


In [23]:
# ==========================================
# 19. Stratified Cross-Validation
# ==========================================

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

print(cv)

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


In [24]:
# ==========================================
# 20. Verify CV Fold Distributions
# ==========================================

cv_distribution_records = []

for fold_number, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train),
    start=1
):

    fold_y_train = y_train.iloc[train_idx]
    fold_y_val = y_train.iloc[val_idx]

    cv_distribution_records.append({
        "Fold": fold_number,
        "Train Churn (%)":
            round(fold_y_train.mean() * 100, 2),
        "Validation Churn (%)":
            round(fold_y_val.mean() * 100, 2)
    })


cv_distribution = pd.DataFrame(
    cv_distribution_records
)

cv_distribution

,Fold,Train Churn (%),Validation Churn (%)
0,1,26.54,26.53
1,2,26.54,26.53
2,3,26.54,26.53
3,4,26.54,26.53
4,5,26.53,26.55


In [25]:
# ==========================================
# 21. Common Evaluation Metrics
# ==========================================

SCORING = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

print("Common Group 4 CV metrics:")

for metric in SCORING:
    print("-", metric)

Common Group 4 CV metrics:
- accuracy
- precision
- recall
- f1
- roc_auc


### Held-Out Test Set Protection

The test set is reserved for final unbiased evaluation.

The following activities will use only the training data together with cross-validation:

- Model development
- Hyperparameter tuning
- Model comparison
- Class-weight experiments
- Feature experiments
- Decision-threshold experiments, if required

Repeatedly checking the test-set result during development would introduce evaluation bias. Therefore, the test set will remain untouched until the final model-selection stage.

## 5. Leakage-Safe Preprocessing Pipeline

Logistic Regression requires numerical model inputs.

Categorical predictors will therefore be one-hot encoded, while numerical predictors will be standardized.

All preprocessing will be fitted inside the machine-learning pipeline so that transformations are learned only from the training portion of each cross-validation fold.

In [26]:
# ==========================================
# 22. Identify Feature Types
# ==========================================

numeric_features = (
    X_train
    .select_dtypes(
        include=["number"]
    )
    .columns
    .tolist()
)

categorical_features = (
    X_train
    .select_dtypes(
        exclude=["number"]
    )
    .columns
    .tolist()
)

print(
    f"Numeric features ({len(numeric_features)}):"
)

for feature in numeric_features:
    print(" -", feature)

print(
    f"\nCategorical features "
    f"({len(categorical_features)}):"
)

for feature in categorical_features:
    print(" -", feature)

Numeric features (19):
 - Senior Citizen
 - Partner
 - Dependents
 - Tenure Months
 - Multiple Lines
 - Online Security
 - Online Backup
 - Device Protection
 - Tech Support
 - Streaming TV
 - Streaming Movies
 - Paperless Billing
 - Payment Method
 - Monthly Charges
 - Total Charges
 - Fiber_Monthly_Risk
 - New_High_Spend
 - New_Monthly_Customer
 - Security_Tech_Bundle

Categorical features (2):
 - Internet Service
 - Contract


In [27]:
# ==========================================
# 23. Feature Type Validation
# ==========================================

identified_features = (
    set(numeric_features)
    | set(categorical_features)
)

expected_features = set(CORE_FEATURES)

missing_from_preprocessing = (
    expected_features
    - identified_features
)

unexpected_features = (
    identified_features
    - expected_features
)

overlap_features = (
    set(numeric_features)
    & set(categorical_features)
)

print(
    "Missing from preprocessing:",
    missing_from_preprocessing
)

print(
    "Unexpected features:",
    unexpected_features
)

print(
    "Numeric/categorical overlap:",
    overlap_features
)

if (
    missing_from_preprocessing
    or unexpected_features
    or overlap_features
):
    raise ValueError(
        "Feature-type assignment validation failed."
    )

print(
    "\nAll core features assigned "
    "exactly once."
)

Missing from preprocessing: set()
Unexpected features: set()
Numeric/categorical overlap: set()

All core features assigned exactly once.


In [28]:
# ==========================================
# 24. Inspect Categorical Levels
# ==========================================

for feature in categorical_features:
    print("=" * 60)
    print(feature)
    print(
        X_train[feature]
        .value_counts(dropna=False)
    )

Internet Service
Internet Service
Fiber optic    2483
DSL            1937
No             1214
Name: count, dtype: int64
Contract
Contract
Month-to-month    3102
Two year          1359
One year          1173
Name: count, dtype: int64


In [29]:
# ==========================================
# 25. Preprocessing Imports
# ==========================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.pipeline import Pipeline

In [30]:
# ==========================================
# 26. Numeric Preprocessing
# ==========================================

numeric_transformer = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [31]:
# ==========================================
# 27. Categorical Preprocessing
# ==========================================

categorical_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [32]:
# ==========================================
# 28. Combined Preprocessor
# ==========================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ],
    remainder="drop"
)

print(preprocessor)

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('scaler', StandardScaler())]),
                                 ['Senior Citizen', 'Partner', 'Dependents',
                                  'Tenure Months', 'Multiple Lines',
                                  'Online Security', 'Online Backup',
                                  'Device Protection', 'Tech Support',
                                  'Streaming TV', 'Streaming Movies',
                                  'Paperless Billing', 'Payment Method',
                                  'Monthly Charges', 'Total Charges',
                                  'Fiber_Monthly_Risk', 'New_High_Spend',
                                  'New_Monthly_Customer',
                                  'Security_Tech_Bundle']),
                                ('categorical',
                                 Pipeline(steps=[('onehot',
                                                  OneHotEncoder(handle_unkn

In [33]:
# ==========================================
# 29. Preprocessing Summary
# ==========================================

print("=" * 60)
print("LOGISTIC REGRESSION PREPROCESSING FOUNDATION")
print("=" * 60)

print(
    f"Numeric features     : "
    f"{len(numeric_features)}"
)

print(
    f"Categorical features : "
    f"{len(categorical_features)}"
)

print(
    f"Total features       : "
    f"{len(numeric_features) + len(categorical_features)}"
)

print(
    "Numeric processing   : StandardScaler"
)

print(
    "Categorical processing: OneHotEncoder"
)

print(
    "Unknown categories   : ignored safely"
)

print(
    "Preprocessing fitted : NO"
)

print(
    "Fit location         : inside CV/model pipeline"
)

print("=" * 60)
print(
    "PREPROCESSOR READY FOR MODEL PIPELINE"
)
print("=" * 60)

LOGISTIC REGRESSION PREPROCESSING FOUNDATION
Numeric features     : 19
Categorical features : 2
Total features       : 21
Numeric processing   : StandardScaler
Categorical processing: OneHotEncoder
Unknown categories   : ignored safely
Preprocessing fitted : NO
Fit location         : inside CV/model pipeline
PREPROCESSOR READY FOR MODEL PIPELINE


## 6. Dummy Classifier Baseline

Before training Logistic Regression, a no-skill baseline is created using `DummyClassifier`.

The baseline provides a reference point for determining whether the machine learning model adds meaningful predictive value.

Because the churn dataset is imbalanced, the majority-class baseline may achieve reasonable accuracy while failing to identify churn customers. Therefore, Precision, Recall, F1-score and ROC-AUC are also evaluated.

In [34]:
# ==========================================
# 30. Dummy Classifier Imports
# ==========================================

from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate

In [35]:
# ==========================================
# 31. Build Dummy Baseline Pipeline
# ==========================================

dummy_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            DummyClassifier(
                strategy="most_frequent"
            )
        )
    ]
)

print(dummy_pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Senior Citizen', 'Partner',
                                                   'Dependents',
                                                   'Tenure Months',
                                                   'Multiple Lines',
                                                   'Online Security',
                                                   'Online Backup',
                                                   'Device Protection',
                                                   'Tech Support',
                                                   'Streaming TV',
                                                   'Streaming Movies',
                                            

In [36]:
# ==========================================
# 32. Cross-Validate Dummy Baseline
# ==========================================

dummy_cv_results = cross_validate(
    estimator=dummy_pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=SCORING,
    return_train_score=False
)

print("DummyClassifier cross-validation completed.")

DummyClassifier cross-validation completed.


d:\Internship Tasks\Group Task - Customer Churn Prediction & Business Intelligence System\Customer-Churn-Prediction-Business-Intelligence-System\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Internship Tasks\Group Task - Customer Churn Prediction & Business Intelligence System\Customer-Churn-Prediction-Business-Intelligence-System\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Internship Tasks\Group Task - Customer Churn Prediction & Business Intelligence System\Customer-Churn-Prediction-

In [37]:
# ==========================================
# 33. Summarize Dummy Baseline Results
# ==========================================

dummy_summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],

    "Mean CV Score": [
        dummy_cv_results["test_accuracy"].mean(),
        dummy_cv_results["test_precision"].mean(),
        dummy_cv_results["test_recall"].mean(),
        dummy_cv_results["test_f1"].mean(),
        dummy_cv_results["test_roc_auc"].mean()
    ],

    "CV Std": [
        dummy_cv_results["test_accuracy"].std(),
        dummy_cv_results["test_precision"].std(),
        dummy_cv_results["test_recall"].std(),
        dummy_cv_results["test_f1"].std(),
        dummy_cv_results["test_roc_auc"].std()
    ]
})

dummy_summary["Mean CV Score"] = (
    dummy_summary["Mean CV Score"]
    .round(4)
)

dummy_summary["CV Std"] = (
    dummy_summary["CV Std"]
    .round(4)
)

dummy_summary

,Metric,Mean CV Score,CV Std
0,Accuracy,0.7346,0.0001
1,Precision,0.0000,0.0000
2,Recall,0.0000,0.0000
3,F1-score,0.0000,0.0000
4,ROC-AUC,0.5000,0.0000


### Dummy Baseline Interpretation

The DummyClassifier acts as a no-skill reference model by predicting the majority class.

Although the baseline may achieve relatively high accuracy because non-churn customers form the majority of the dataset, its ability to identify churn customers is extremely poor.

This demonstrates why accuracy alone is not an appropriate model-selection criterion for this churn problem.

The Logistic Regression model should therefore be evaluated based on its improvement over this baseline, particularly in terms of Recall, F1-score and ROC-AUC.

## 7. Logistic Regression Baseline

Logistic Regression is used as the first real classification model for customer churn prediction.

Unlike the DummyClassifier, Logistic Regression learns relationships between customer features and the probability of churn.

The model is evaluated using the same preprocessing pipeline, stratified 5-fold cross-validation strategy and evaluation metrics used for the baseline so that the comparison remains fair.

The held-out test set is not used during this stage.

In [38]:
# ==========================================
# 34. Logistic Regression Import
# ==========================================

from sklearn.linear_model import LogisticRegression

In [39]:
# ==========================================
# 35. Build Baseline Logistic Regression
# ==========================================

logistic_baseline = LogisticRegression(
    max_iter=2000,
    random_state=RANDOM_STATE
)

print(logistic_baseline)

LogisticRegression(max_iter=2000, random_state=42)


In [40]:
# ==========================================
# 36. Build Logistic Regression Pipeline
# ==========================================

logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            logistic_baseline
        )
    ]
)

print(logistic_pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Senior Citizen', 'Partner',
                                                   'Dependents',
                                                   'Tenure Months',
                                                   'Multiple Lines',
                                                   'Online Security',
                                                   'Online Backup',
                                                   'Device Protection',
                                                   'Tech Support',
                                                   'Streaming TV',
                                                   'Streaming Movies',
                                            

In [41]:
# ==========================================
# 37. Cross-Validate Logistic Regression
# ==========================================

logistic_cv_results = cross_validate(
    estimator=logistic_pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=SCORING,
    return_train_score=True,
    n_jobs=-1
)

print(
    "Logistic Regression cross-validation completed."
)

Logistic Regression cross-validation completed.


In [42]:
# ==========================================
# 38. Logistic Regression Fold Results
# ==========================================

logistic_fold_results = pd.DataFrame({
    "Fold": range(1, N_SPLITS + 1),

    "Accuracy": (
        logistic_cv_results["test_accuracy"]
    ),

    "Precision": (
        logistic_cv_results["test_precision"]
    ),

    "Recall": (
        logistic_cv_results["test_recall"]
    ),

    "F1-score": (
        logistic_cv_results["test_f1"]
    ),

    "ROC-AUC": (
        logistic_cv_results["test_roc_auc"]
    )
})

numeric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "ROC-AUC"
]

logistic_fold_results[numeric_columns] = (
    logistic_fold_results[numeric_columns]
    .round(4)
)

logistic_fold_results

,Fold,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,1,0.8083,0.6615,0.5686,0.6115,0.8636
1,2,0.7941,0.6356,0.5251,0.5751,0.8385
2,3,0.8128,0.6746,0.5686,0.6171,0.8535
3,4,0.8243,0.6836,0.6288,0.6551,0.8711
4,5,0.8224,0.7162,0.5485,0.6212,0.8717


In [43]:
# ==========================================
# 39. Logistic Regression CV Summary
# ==========================================

logistic_summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],

    "Mean CV Score": [
        logistic_cv_results[
            "test_accuracy"
        ].mean(),

        logistic_cv_results[
            "test_precision"
        ].mean(),

        logistic_cv_results[
            "test_recall"
        ].mean(),

        logistic_cv_results[
            "test_f1"
        ].mean(),

        logistic_cv_results[
            "test_roc_auc"
        ].mean()
    ],

    "CV Std": [
        logistic_cv_results[
            "test_accuracy"
        ].std(),

        logistic_cv_results[
            "test_precision"
        ].std(),

        logistic_cv_results[
            "test_recall"
        ].std(),

        logistic_cv_results[
            "test_f1"
        ].std(),

        logistic_cv_results[
            "test_roc_auc"
        ].std()
    ]
})

logistic_summary[
    ["Mean CV Score", "CV Std"]
] = (
    logistic_summary[
        ["Mean CV Score", "CV Std"]
    ]
    .round(4)
)

logistic_summary

,Metric,Mean CV Score,CV Std
0,Accuracy,0.8124,0.0109
1,Precision,0.6743,0.0265
2,Recall,0.5679,0.0344
3,F1-score,0.6160,0.0255
4,ROC-AUC,0.8597,0.0125


In [44]:
# ==========================================
# 40. Training vs Validation Performance
# ==========================================

generalization_check = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],

    "Mean Train Score": [
        logistic_cv_results[
            "train_accuracy"
        ].mean(),

        logistic_cv_results[
            "train_precision"
        ].mean(),

        logistic_cv_results[
            "train_recall"
        ].mean(),

        logistic_cv_results[
            "train_f1"
        ].mean(),

        logistic_cv_results[
            "train_roc_auc"
        ].mean()
    ],

    "Mean Validation Score": [
        logistic_cv_results[
            "test_accuracy"
        ].mean(),

        logistic_cv_results[
            "test_precision"
        ].mean(),

        logistic_cv_results[
            "test_recall"
        ].mean(),

        logistic_cv_results[
            "test_f1"
        ].mean(),

        logistic_cv_results[
            "test_roc_auc"
        ].mean()
    ]
})

generalization_check["Gap"] = (
    generalization_check["Mean Train Score"]
    - generalization_check[
        "Mean Validation Score"
    ]
)

generalization_check[
    [
        "Mean Train Score",
        "Mean Validation Score",
        "Gap"
    ]
] = (
    generalization_check[
        [
            "Mean Train Score",
            "Mean Validation Score",
            "Gap"
        ]
    ]
    .round(4)
)

generalization_check

,Metric,Mean Train Score,Mean Validation Score,Gap
0,Accuracy,0.8160,0.8124,0.0036
1,Precision,0.6816,0.6743,0.0073
2,Recall,0.5754,0.5679,0.0075
3,F1-score,0.6240,0.6160,0.0080
4,ROC-AUC,0.8635,0.8597,0.0038


In [45]:
# ==========================================
# 41. Dummy vs Logistic Regression
# ==========================================

baseline_comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],

    "DummyClassifier": [
        dummy_cv_results[
            "test_accuracy"
        ].mean(),

        dummy_cv_results[
            "test_precision"
        ].mean(),

        dummy_cv_results[
            "test_recall"
        ].mean(),

        dummy_cv_results[
            "test_f1"
        ].mean(),

        dummy_cv_results[
            "test_roc_auc"
        ].mean()
    ],

    "Logistic Regression": [
        logistic_cv_results[
            "test_accuracy"
        ].mean(),

        logistic_cv_results[
            "test_precision"
        ].mean(),

        logistic_cv_results[
            "test_recall"
        ].mean(),

        logistic_cv_results[
            "test_f1"
        ].mean(),

        logistic_cv_results[
            "test_roc_auc"
        ].mean()
    ]
})

baseline_comparison[
    [
        "DummyClassifier",
        "Logistic Regression"
    ]
] = (
    baseline_comparison[
        [
            "DummyClassifier",
            "Logistic Regression"
        ]
    ]
    .round(4)
)

baseline_comparison

,Metric,DummyClassifier,Logistic Regression
0,Accuracy,0.7346,0.8124
1,Precision,0.0000,0.6743
2,Recall,0.0000,0.5679
3,F1-score,0.0000,0.6160
4,ROC-AUC,0.5000,0.8597


### Logistic Regression Baseline Interpretation

The baseline Logistic Regression model is compared against the no-skill DummyClassifier using the same stratified 5-fold cross-validation procedure.

The comparison focuses on Accuracy, Precision, Recall, F1-score and ROC-AUC rather than Accuracy alone.

Particular attention is given to Recall because a false negative represents a customer who actually churns but is predicted as non-churn. In a retention setting, these customers may not receive an intervention.

Cross-validation variability and the difference between training and validation performance are also examined to assess the stability and generalization of the Logistic Regression model.

The baseline Logistic Regression results will next be used as the reference for hyperparameter tuning.

## 8. Logistic Regression Hyperparameter Tuning

The baseline Logistic Regression model is now tuned using cross-validation.

Hyperparameter tuning is performed only on the training data. The held-out test set remains untouched.

The search evaluates different regularization strengths, regularization types, solver choices and class-weight settings.

F1-score is used as the primary refit metric because the churn problem requires a balance between Precision and Recall. All other evaluation metrics are also retained for comparison.

In [46]:
# ==========================================
# 42. Logistic Regression Tuning Imports
# ==========================================

from sklearn.model_selection import GridSearchCV

In [47]:
# ==========================================
# 43. Logistic Regression Parameter Grid
# ==========================================

logistic_param_grid = [
    {
        "classifier__solver": ["lbfgs"],
        "classifier__penalty": ["l2"],
        "classifier__C": [
            0.01,
            0.1,
            1.0,
            10.0,
            100.0
        ],
        "classifier__class_weight": [
            None,
            "balanced"
        ]
    },

    {
        "classifier__solver": ["liblinear"],
        "classifier__penalty": [
            "l1",
            "l2"
        ],
        "classifier__C": [
            0.01,
            0.1,
            1.0,
            10.0,
            100.0
        ],
        "classifier__class_weight": [
            None,
            "balanced"
        ]
    }
]

print("Logistic Regression tuning grid created.")

Logistic Regression tuning grid created.


In [48]:
# ==========================================
# 44. Configure Logistic Grid Search
# ==========================================

logistic_grid_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_param_grid,
    scoring=SCORING,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
    verbose=1
)

print(logistic_grid_search)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('numeric',
                                                                         Pipeline(steps=[('scaler',
                                                                                          StandardScaler())]),
                                                                         ['Senior '
                                                                          'Citizen',
                                                                          'Partner',
                                                                          'Dependents',
                                                                          'Tenure '
                                                                          'Months',
                                                               

In [49]:
# ==========================================
# 45. Run Logistic Regression Grid Search
# ==========================================

logistic_grid_search.fit(
    X_train,
    y_train
)

print(
    "Logistic Regression hyperparameter "
    "tuning completed."
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Logistic Regression hyperparameter tuning completed.


d:\Internship Tasks\Group Task - Customer Churn Prediction & Business Intelligence System\Customer-Churn-Prediction-Business-Intelligence-System\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\Internship Tasks\Group Task - Customer Churn Prediction & Business Intelligence System\Customer-Churn-Prediction-Business-Intelligence-System\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [50]:
# ==========================================
# 46. Best Logistic Regression Parameters
# ==========================================

print("Best parameters:")

for parameter, value in (
    logistic_grid_search
    .best_params_
    .items()
):
    print(
        f" - {parameter}: {value}"
    )

print(
    "\nBest mean CV F1-score:",
    round(
        logistic_grid_search.best_score_,
        4
    )
)

Best parameters:
 - classifier__C: 100.0
 - classifier__class_weight: balanced
 - classifier__penalty: l1
 - classifier__solver: liblinear

Best mean CV F1-score: 0.6446


In [51]:
# ==========================================
# 47. Best Tuned Logistic CV Summary
# ==========================================

best_index = logistic_grid_search.best_index_

tuned_logistic_summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],

    "Mean CV Score": [
        logistic_grid_search.cv_results_[
            "mean_test_accuracy"
        ][best_index],

        logistic_grid_search.cv_results_[
            "mean_test_precision"
        ][best_index],

        logistic_grid_search.cv_results_[
            "mean_test_recall"
        ][best_index],

        logistic_grid_search.cv_results_[
            "mean_test_f1"
        ][best_index],

        logistic_grid_search.cv_results_[
            "mean_test_roc_auc"
        ][best_index]
    ],

    "CV Std": [
        logistic_grid_search.cv_results_[
            "std_test_accuracy"
        ][best_index],

        logistic_grid_search.cv_results_[
            "std_test_precision"
        ][best_index],

        logistic_grid_search.cv_results_[
            "std_test_recall"
        ][best_index],

        logistic_grid_search.cv_results_[
            "std_test_f1"
        ][best_index],

        logistic_grid_search.cv_results_[
            "std_test_roc_auc"
        ][best_index]
    ]
})

tuned_logistic_summary[
    ["Mean CV Score", "CV Std"]
] = (
    tuned_logistic_summary[
        ["Mean CV Score", "CV Std"]
    ]
    .round(4)
)

tuned_logistic_summary

,Metric,Mean CV Score,CV Std
0,Accuracy,0.7622,0.0151
1,Precision,0.5341,0.0190
2,Recall,0.8134,0.0368
3,F1-score,0.6446,0.0237
4,ROC-AUC,0.8596,0.0124


In [52]:
# ==========================================
# 48. Tuned Logistic Generalization Check
# ==========================================

tuned_generalization = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],

    "Mean Train Score": [
        logistic_grid_search.cv_results_[
            "mean_train_accuracy"
        ][best_index],

        logistic_grid_search.cv_results_[
            "mean_train_precision"
        ][best_index],

        logistic_grid_search.cv_results_[
            "mean_train_recall"
        ][best_index],

        logistic_grid_search.cv_results_[
            "mean_train_f1"
        ][best_index],

        logistic_grid_search.cv_results_[
            "mean_train_roc_auc"
        ][best_index]
    ],

    "Mean Validation Score": [
        logistic_grid_search.cv_results_[
            "mean_test_accuracy"
        ][best_index],

        logistic_grid_search.cv_results_[
            "mean_test_precision"
        ][best_index],

        logistic_grid_search.cv_results_[
            "mean_test_recall"
        ][best_index],

        logistic_grid_search.cv_results_[
            "mean_test_f1"
        ][best_index],

        logistic_grid_search.cv_results_[
            "mean_test_roc_auc"
        ][best_index]
    ]
})

tuned_generalization["Gap"] = (
    tuned_generalization["Mean Train Score"]
    - tuned_generalization[
        "Mean Validation Score"
    ]
)

tuned_generalization[
    [
        "Mean Train Score",
        "Mean Validation Score",
        "Gap"
    ]
] = (
    tuned_generalization[
        [
            "Mean Train Score",
            "Mean Validation Score",
            "Gap"
        ]
    ]
    .round(4)
)

tuned_generalization

,Metric,Mean Train Score,Mean Validation Score,Gap
0,Accuracy,0.7638,0.7622,0.0016
1,Precision,0.5361,0.5341,0.0021
2,Recall,0.8157,0.8134,0.0023
3,F1-score,0.6470,0.6446,0.0024
4,ROC-AUC,0.8635,0.8596,0.0039


In [53]:
# ==========================================
# 49. Baseline vs Tuned Logistic Regression
# ==========================================

logistic_tuning_comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],

    "Baseline Logistic": [
        logistic_cv_results[
            "test_accuracy"
        ].mean(),

        logistic_cv_results[
            "test_precision"
        ].mean(),

        logistic_cv_results[
            "test_recall"
        ].mean(),

        logistic_cv_results[
            "test_f1"
        ].mean(),

        logistic_cv_results[
            "test_roc_auc"
        ].mean()
    ],

    "Tuned Logistic": [
        logistic_grid_search.cv_results_[
            "mean_test_accuracy"
        ][best_index],

        logistic_grid_search.cv_results_[
            "mean_test_precision"
        ][best_index],

        logistic_grid_search.cv_results_[
            "mean_test_recall"
        ][best_index],

        logistic_grid_search.cv_results_[
            "mean_test_f1"
        ][best_index],

        logistic_grid_search.cv_results_[
            "mean_test_roc_auc"
        ][best_index]
    ]
})

logistic_tuning_comparison[
    "Difference"
] = (
    logistic_tuning_comparison[
        "Tuned Logistic"
    ]
    - logistic_tuning_comparison[
        "Baseline Logistic"
    ]
)

logistic_tuning_comparison[
    [
        "Baseline Logistic",
        "Tuned Logistic",
        "Difference"
    ]
] = (
    logistic_tuning_comparison[
        [
            "Baseline Logistic",
            "Tuned Logistic",
            "Difference"
        ]
    ]
    .round(4)
)

logistic_tuning_comparison

,Metric,Baseline Logistic,Tuned Logistic,Difference
0,Accuracy,0.8124,0.7622,-0.0502
1,Precision,0.6743,0.5341,-0.1402
2,Recall,0.5679,0.8134,0.2455
3,F1-score,0.6160,0.6446,0.0286
4,ROC-AUC,0.8597,0.8596,-0.0001


In [54]:
# ==========================================
# 50. Top Logistic Regression Configurations
# ==========================================

logistic_search_results = pd.DataFrame(
    logistic_grid_search.cv_results_
)

top_logistic_configs = (
    logistic_search_results[
        [
            "params",
            "mean_test_accuracy",
            "mean_test_precision",
            "mean_test_recall",
            "mean_test_f1",
            "mean_test_roc_auc",
            "std_test_f1"
        ]
    ]
    .sort_values(
        by="mean_test_f1",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)

score_columns = [
    "mean_test_accuracy",
    "mean_test_precision",
    "mean_test_recall",
    "mean_test_f1",
    "mean_test_roc_auc",
    "std_test_f1"
]

top_logistic_configs[
    score_columns
] = (
    top_logistic_configs[
        score_columns
    ]
    .round(4)
)

top_logistic_configs

,params,mean_test_accuracy,mean_test_precision,mean_test_recall,mean_test_f1,mean_test_roc_auc,std_test_f1
0,"{'classifier__C': 100.0, 'classifier__class_we...",0.7622,0.5341,0.8134,0.6446,0.8596,0.0237
1,"{'classifier__C': 100.0, 'classifier__class_we...",0.7616,0.5334,0.8134,0.6441,0.8595,0.0244
2,"{'classifier__C': 100.0, 'classifier__class_we...",0.7618,0.5336,0.8127,0.6441,0.8596,0.0241
3,"{'classifier__C': 1.0, 'classifier__class_weig...",0.7622,0.5341,0.8114,0.6440,0.8595,0.0233
4,"{'classifier__C': 10.0, 'classifier__class_wei...",0.7616,0.5334,0.8127,0.6439,0.8595,0.0251
5,"{'classifier__C': 10.0, 'classifier__class_wei...",0.7618,0.5336,0.8120,0.6439,0.8596,0.0248
6,"{'classifier__C': 10.0, 'classifier__class_wei...",0.7616,0.5334,0.8120,0.6437,0.8596,0.0247
7,"{'classifier__C': 1.0, 'classifier__class_weig...",0.7616,0.5334,0.8114,0.6435,0.8595,0.0242
8,"{'classifier__C': 1.0, 'classifier__class_weig...",0.7616,0.5334,0.8114,0.6435,0.8595,0.0242
9,"{'classifier__C': 0.01, 'classifier__class_wei...",0.7639,0.5377,0.7953,0.6414,0.8561,0.0232


### Hyperparameter Tuning Interpretation

The Logistic Regression model was tuned using stratified 5-fold cross-validation on the training data only.

The search evaluated regularization strength (`C`), regularization type, solver choice and class weighting.

F1-score was used to identify the best configuration because the churn problem requires a balance between Precision and Recall.

The tuned model is compared against the baseline Logistic Regression across all required evaluation metrics. Particular attention is given to changes in Recall, Precision and F1-score rather than selecting a model based only on Accuracy.

Training and validation scores are also compared to ensure that the selected configuration generalizes consistently across the cross-validation folds.

The held-out test set remains unused during hyperparameter tuning.